# Spotify dataset — reproducible validation and preprocessing

This notebook runs against the repository's included cleaned dataset by default. It can also preprocess the **original raw dataset** if you supply that file separately.

**Important:** When the cleaned dataset is selected, the notebook **does not reapply outlier capping or other transformations**, because doing so could change the published analysis. The original coursework notebook is preserved as [`original_etl_coursework.ipynb`](original_etl_coursework.ipynb).

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

# Works when launched from the repository root or notebooks/ directory.
root = Path.cwd()
if not (root / 'data' / 'spotify_cleaned.csv').exists():
    root = root.parent
cleaned_path = root / 'data' / 'spotify_cleaned.csv'
assert cleaned_path.is_file(), f'Cannot find {cleaned_path}. Run this notebook from the repository or notebooks folder.'

# Optional: set this to a local copy of the ORIGINAL raw Spotify dataset.
# Example: raw_path = root / 'data' / 'spotify_raw.csv'
raw_path = None
source_path = Path(raw_path) if raw_path is not None else cleaned_path
print('Input:', source_path)
df = pd.read_csv(source_path, encoding='utf-8-sig')
print('Rows:', len(df), '| Columns:', len(df.columns))
df.head()

Input: /workspaces/spotify-music-trends-analysis/data/spotify_cleaned.csv
Rows: 11762 | Columns: 20


,valence,year,acousticness,artists,danceability,energy,explicit,id,instrumentalness,key,liveness,loudness,mode,name,popularity,release_date,speechiness,tempo,Genre,duration_min
0,0.518,2015,0.005,['BØRNS'],0.611,0.797,0,2GiJYvgVaD2HtM8GqD9EgQ,0.001,6,0.260,-7.627,0,Electric Love,86.5,16/10/2015,0.053,120.041,Indie Pop / Dream Pop,3.635
1,0.337,2015,0.068,['The Neighbourhood'],0.588,0.521,0,5E30LdtzQTGqRvNd7l6kG5,0.001,10,0.123,-60.000,1,Daddy Issues,86.0,30/10/2015,0.033,85.012,Alternative R&B,4.336
2,0.785,2015,0.012,['Tame Impala'],0.640,0.740,1,6K4t31amVTZDgR3sKmwUJJ,0.001,4,0.167,-60.000,1,The Less I Know The Better,85.0,17/07/2015,0.028,116.879,Psychedelic Pop / Indie Rock,3.605
3,0.512,2015,0.205,['Chris Stapleton'],0.392,0.370,0,3fqwjXwUGN6vbzIwvyFMhx,0.001,9,0.082,-60.000,1,Tennessee Whiskey,79.0,04/05/2015,0.030,50.000,Country Soul,4.888
4,0.672,2015,0.037,['WILLOW'],0.764,0.705,0,0y60itmpH0aPKsFiGxmtnh,0.000,3,0.094,-60.000,0,Wait a Minute!,83.0,11/01/2015,0.028,101.003,Alternative R&B / Pop,3.275


## 1. Data quality overview

Inspect missing values, duplicates, column types, and the expected columns before making any changes.

In [2]:
print('Data types:')
print(df.dtypes.to_string())
print('\nMissing values:')
print(df.isna().sum().to_string())
print('\nDuplicate rows:', int(df.duplicated().sum()))
required = {'id', 'name', 'artists', 'popularity', 'release_date'}
missing_required = required - set(df.columns)
assert not missing_required, f'Missing required fields: {sorted(missing_required)}'

Data types:
valence             float64
year                  int64
acousticness        float64
artists                 str
danceability        float64
energy              float64
explicit              int64
id                      str
instrumentalness    float64
key                   int64
liveness            float64
loudness            float64
mode                  int64
name                    str
popularity          float64
release_date            str
speechiness         float64
tempo               float64
Genre                   str
duration_min        float64

Missing values:
valence             0
year                0
acousticness        0
artists             0
danceability        0
energy              0
explicit            0
id                  0
instrumentalness    0
key                 0
liveness            0
loudness            0
mode                0
name                0
popularity          0
release_date        0
speechiness         0
tempo               0
Genre          

## 2. Prepare raw data (only when an original raw file is provided)

The original coursework workflow converted dates and duration, removed the original duration column, and applied IQR-based capping to numeric columns. Those steps are intentionally **not rerun** on the already-cleaned portfolio CSV.

In [3]:
is_raw = raw_path is not None
if is_raw:
    if 'duration_ms' not in df.columns:
        raise ValueError('Raw preprocessing requires a duration_ms column. Do not use the cleaned CSV as raw input.')
    df['release_date'] = pd.to_datetime(df['release_date'], errors='coerce', dayfirst=True)
    df['duration_min'] = pd.to_numeric(df['duration_ms'], errors='coerce') / 60000
    df = df.drop(columns=['duration_ms'])

    numeric_cols = df.select_dtypes(include=np.number).columns
    for col in numeric_cols:
        q1, q3 = df[col].quantile([0.25, 0.75])
        iqr = q3 - q1
        if pd.notna(iqr) and iqr > 0:
            df[col] = df[col].clip(lower=q1 - 1.5 * iqr, upper=q3 + 1.5 * iqr)
    print('Raw preprocessing completed; review changes before using in Power BI.')
else:
    print('Using published cleaned dataset: no transformations applied.')

Using published cleaned dataset: no transformations applied.


## 3. Validation and summary

The summary is computed on the selected dataset. The published cleaned CSV remains unchanged.

In [4]:
assert 'duration_min' in df.columns, 'Expected duration_min column.'
assert 'Genre' in df.columns, 'Expected Genre column.'
print('Shape:', df.shape)
print('Unique track IDs:', df['id'].nunique(dropna=True))
print('Popularity summary:')
print(pd.to_numeric(df['popularity'], errors='coerce').describe())
print('Duration (minutes) summary:')
print(pd.to_numeric(df['duration_min'], errors='coerce').describe())
print('Missing values (total):', int(df.isna().sum().sum()))

Shape: (11762, 20)
Unique track IDs: 11762
Popularity summary:
count    11762.000000
mean        64.234994
std          9.324067
min         42.500000
25%         59.000000
50%         64.000000
75%         70.000000
max         86.500000
Name: popularity, dtype: float64
Duration (minutes) summary:
count    11762.000000
mean         3.459434
std          0.820929
min          1.518000
25%          2.955000
50%          3.423000
75%          3.910000
max          5.346000
Name: duration_min, dtype: float64
Missing values (total): 0


## 4. Optional export

Export only when processing an original raw dataset. This avoids overwriting the portfolio's published CSV by accident.

In [5]:
if is_raw:
    output_path = root / 'data' / 'spotify_preprocessed_from_raw.csv'
    df.to_csv(output_path, index=False)
    print('Wrote:', output_path)
else:
    print('No export: the published cleaned dataset was inspected without modification.')

No export: the published cleaned dataset was inspected without modification.


## Notes and limitations

- The repository does **not** contain the original raw dataset required to regenerate the exact published CSV from scratch.
- The coursework notebook used a machine-specific Windows input path and contained a final `Genr` assignment sourced from `tempo`; this version avoids that assignment.
- IQR capping can alter values, so the raw-input workflow should be checked against the original coursework methodology before comparing results with the existing dashboard.
- This notebook documents and validates data preparation; it does not rebuild the SQL star schema or Power BI report.